# SSD300 Object Detection with PyTorch and TorchVision


we will implement the SSD300 (Single Shot MultiBox Detector) model using the PyTorch and TorchVision libraries.

The goal of this model is to perform Object Detection; meaning that, in addition to recognizing the class of an object, it also determines its location by predicting its Bounding Box.

In this session, we will cover the following topics:

- Preparing the Dataset for SSD
- Building the DataLoader
- Using the pre-trained SSD300-VGG16 model
- Fine-tuning the model with pre-trained weights
- Modifying the Classification Head for a new dataset
- Defining the Optimizer
- Training the model
- Using Validation Loss to monitor the training process
- Evaluating the model using the mAP metric
- Performing Prediction on new images

In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, random_split
import torchvision

In [ ]:
torch.cuda.is_available()

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DEVICE

In [ ]:
IMAGE_DIR = 'dataset/images'
ANNOTATIONS_DIR = 'dataset/annotations'
TARGET_SIZE = (300, 300)
BATCH_SIZE = 32

# Load Dataset

In [ ]:
class_names = [d for d in os.listdir(IMAGE_DIR) if os.path.isdir(os.path.join(IMAGE_DIR, d))]

In [ ]:
class_names

In [ ]:
csv_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.csv')]

In [ ]:
csv_files

In [ ]:
label_map = {name: i + 1 for i, name in enumerate(class_names)}

In [ ]:
label_map

In [ ]:
dataset = []

for i in range(len(class_names)):
    class_name = class_names[i]
    class_dir = os.path.join(IMAGE_DIR, class_name)
    csv_file_name = csv_files[i]
    
    csv_path = os.path.join(ANNOTATIONS_DIR, csv_file_name)
    df_annotations = pd.read_csv(csv_path)

    for image_name in os.listdir(class_dir):
        image_path = os.path.join(class_dir, image_name)
        
        image = cv2.imread(image_path)
        h, w, _ = image.shape
        
        ann = df_annotations[df_annotations['image_name'] == image_name].iloc[0,1:].tolist()

        if (ann[2] > ann[0] and ann[3] > ann[1]):
            ann[0] = int((ann[0] / w) * 300)
            ann[1] = int((ann[1] / h) * 300)
            ann[2] = int((ann[2] / w) * 300)
            ann[3] = int((ann[3] / h) * 300)
            
            
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image_resized = cv2.resize(image, TARGET_SIZE)
            
            image_tensor = torch.tensor(image_resized, dtype=torch.float32).permute(2, 0, 1) / 255.0
            
            class_id = label_map[class_name]
            labels_tensor = torch.tensor([class_id], dtype=torch.int64)
            ann_tensor = torch.tensor([ann], dtype=torch.float32)
    
            target = {
                'boxes': ann_tensor,
                'labels': labels_tensor
            }
            dataset.append((image_tensor, target))
        else:
            print(f"️Invalid box found and removed in '{image_path}': {ann}")

def collate_fn(batch):
    return tuple(zip(*batch))

# Train_Test_Split

In [ ]:
train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size

In [ ]:
generator = torch.Generator().manual_seed(42)

train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=generator)

In [ ]:
print(f"Dataset ({len(dataset)})")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train ({len(train_dataset)}), Test ({len(test_dataset)})")

# Modeling

### Using SSD300 Model in TorchVision

The **TorchVision** library provides pre-trained **Object Detection** models.

The model used in this project is:

```text
SSD300 + VGG16 Backbone
````

In TorchVision, it can be created using:

```python
ssd300_vgg16()
```

```


# Using Pretrained Weights

The SSD300 model can be used in two different ways:

## Training from Scratch

```text
Random Weights
        |
        |
     Training
````

## Using Pretrained Weights

```text
  COCO Dataset
        |
        |
  Pretrained SSD
        |
        |
   Fine-tuning
```

In this project, the second approach is used.

Advantages:

* Faster training
* Requires less data
* Better performance

```


In [ ]:
model = torchvision.models.detection.ssd300_vgg16(
    weights=torchvision.models.detection.SSD300_VGG16_Weights.DEFAULT,
)


# Changing the Number of Classes in SSD

The SSD model pretrained on COCO contains **91 classes**:

```

90 Object Classes + Background = 91 Classes

```

However, a custom dataset may contain a different number of classes:

```

3 Object Classes + Background = 4 Classes

```

Therefore, the model's **Classification Head** must be replaced.

Directly changing `num_classes` during weight loading is not possible because the pretrained Head weights are designed for 91 classes and are incompatible with the new output size.

Solution:
- Keep the pretrained **Backbone** weights
- Replace the **Classification Head** with a new Head matching the dataset classes
```



# Changing the Classification Head in SSD

The SSD model has two output branches:

```

SSD Head

```

  |
  +---- Classification Head  →  Class prediction
  |
  +---- Regression Head      →  Bounding box prediction

```

```

For a new dataset, only the **Classification Head** needs to be replaced because the number of classes changes.

The **Regression Head** remains unchanged since bounding box prediction keeps the same structure.

---

# Creating a New Head

To create a new Classification Head, three values are required:

### 1. Input Channels

The number of input Feature Map channels, obtained from the existing model.

### 2. Number of Anchors

The number of anchors assigned to each Feature Map, obtained from the model's **Anchor Generator**.

Example:

```

Feature Map 1 → 4 Anchors
Feature Map 2 → 6 Anchors

```

### 3. Number of Classes

The number of classes in the new dataset:

```

Object Classes + Background

```

Example:

```

3 Objects + 1 Background = 4 Classes

```
```


In [ ]:
in_channels = []
for i in range(len(model.head.classification_head.module_list)):
    in_channels.append(model.head.classification_head.module_list[i].in_channels)

In [ ]:
in_channels

In [ ]:
num_anchors = model.anchor_generator.num_anchors_per_location()

In [ ]:
num_anchors

In [ ]:
num_classes = len(label_map) + 1

In [ ]:
model.head.classification_head = torchvision.models.detection.ssd.SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes
)


# Fine-tuning Model

After replacing the Head, the model is ready for Fine-tuning.

During this process:

- The **Backbone** keeps the pretrained weights and extracts image features.
- The **new Head** is trained according to the classes of the new dataset.

The goal of Fine-tuning is to adapt the SSD model to the new dataset.
```


In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)

In [ ]:
epochs = 5
model.to(DEVICE)
model.train()

for epoch in range(epochs):
    train_loss = 0.0
    # ---- Train ----
    for images, targets in train_loader:
        optimizer.zero_grad()
        
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        
        loss_dict_train = model(images, targets)
        loss = sum(l for l in loss_dict_train.values())
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

     # ---- Validation ----
    test_loss = 0.0
    with torch.no_grad():
        for images, targets in test_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            
            loss_dict_test = model(images, targets)
            loss = sum(l for l in loss_dict_test.values())
            test_loss += loss.item()
            
    avg_train_loss = train_loss / len(train_loader)
    avg_test_loss = test_loss / len(test_loader)
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_test_loss:.4f}")

# Calculate mAP

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [ ]:
def calculate_map(model, dataloader, device):
    metric = MeanAveragePrecision(box_format='xyxy')
    
    model.eval()
    model.to(device)
    
    with torch.no_grad():
        for images, targets in dataloader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            predictions = model(images)
            
            metric.update(predictions, targets)
            
    results = metric.compute()
    

    print(f"mAP: {results['map'].item():.4f}")
    print(f"mAP@50 (IoU=0.50): {results['map_50'].item():.4f}")
    print(f"mAP@75 (IoU=0.75): {results['map_75'].item():.4f}")

    return results

In [ ]:
test_map_results = calculate_map(model, test_loader, DEVICE)

In [ ]:
train_map_results = calculate_map(model, train_loader, DEVICE)

In [ ]:
def predict_on_image_single_object(model, image_path, device, label_map):

    original_image = cv2.imread(image_path)
    original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    
    image_for_model = cv2.resize(original_image_rgb, TARGET_SIZE)
    image_tensor = (torch.tensor(image_for_model, dtype=torch.float32).permute(2, 0, 1) / 255.0).unsqueeze(0)
    
    model.eval()
    model.to(device)

    with torch.no_grad():
        prediction = model(image_tensor.to(device))[0]
        
    if len(prediction['scores']) == 0:
        print("No object")
        return

    best_score_idx = torch.argmax(prediction['scores']).item()
    score = prediction['scores'][best_score_idx].cpu().item()
    box = prediction['boxes'][best_score_idx].cpu().numpy()
    label_id = prediction['labels'][best_score_idx].cpu().item()


    reverse_label_map = {v: k for k, v in label_map.items()}
    label_name = reverse_label_map.get(label_id, 'Unknown')

    h_orig, w_orig, _ = original_image.shape
    x_scale = w_orig / TARGET_SIZE[0]
    y_scale = h_orig / TARGET_SIZE[1]
    
    xmin = int(box[0] * x_scale)
    ymin = int(box[1] * y_scale)
    xmax = int(box[2] * x_scale)
    ymax = int(box[3] * y_scale)
    
    cv2.rectangle(original_image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 3) 
    cv2.putText(original_image, f"{label_name}: {score:.2f}", (xmin, ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

In [ ]:
image_path_to_predict = "dataset/images/face/image_0002.jpg"
predict_on_image_single_object(model, image_path_to_predict, DEVICE, label_map)


# Summary

Complete Object Detection pipeline with SSD300:

```

Dataset
|
DataLoader
|
SSD300-VGG16
|
Replace Classification Head
|
Fine-tuning
|
Validation Monitoring
|
mAP Evaluation
|
Prediction

```

Key points:

- Using a **Pretrained Model** reduces training time and improves performance.
- In **Transfer Learning**, the model Head is usually replaced to match the new dataset.
- **Validation** is essential for monitoring and preventing Overfitting.
- The main evaluation metric in Object Detection is **mAP**.
```
